# Worked example: from bbox to animal GPS

This notebook walks through the geolocation math end-to-end on a single synthetic detection. It depends only on the `camera_trap_geolocation` package - no GPU, no model weights - so you can run it anywhere.

Install in editable mode from the repo root:

```bash
pip install -e .
```

In [ ]:
from camera_trap_geolocation import (
    estimate_distance_and_angle,
    project_gps,
    focal_length_px,
)

## Scenario

A 1920x1080 trail-camera frame with a 60-degree horizontal field of view. A bounding box appears at (810, 400) to (1110, 700) - centered horizontally, 300 pixels tall. The target species is a white-tailed deer (about 0.9 m at the shoulder).

The camera was deployed at 40.123 N, 83.456 W and the compass bearing of the lens is 270 degrees (due west).

In [ ]:
img_w, img_h = 1920, 1080
x_min, y_min, x_max, y_max = 810, 400, 1110, 700

hfov_deg = 60.0
real_height_m = 0.9
camera_lat = 40.123
camera_lon = -83.456
camera_bearing_deg = 270.0

fx = focal_length_px(img_w, hfov_deg)
print(f'Focal length: {fx:.2f} px')

In [ ]:
distance_m, angle_deg = estimate_distance_and_angle(
    img_w, img_h, x_min, y_min, x_max, y_max,
    hfov_deg=hfov_deg,
    real_height_m=real_height_m,
)
print(f'Distance to deer: {distance_m:.2f} m')
print(f'Relative angle:   {angle_deg:+.2f} deg')

In [ ]:
absolute_bearing_deg = (camera_bearing_deg + angle_deg) % 360.0
animal_lat, animal_lon = project_gps(
    camera_lat, camera_lon,
    distance_m, absolute_bearing_deg,
)
print(f'Absolute bearing to deer: {absolute_bearing_deg:.2f} deg')
print(f'Estimated deer GPS:       {animal_lat:.6f}, {animal_lon:.6f}')

## Sanity check

The bbox is centered horizontally, so the relative angle should be near 0 degrees. The camera bearing is 270 (due west), so the deer should sit roughly due west of the camera - same latitude, smaller (more negative) longitude.

## Try a different bearing

Change `camera_bearing_deg` to 180 (south) and rerun: now the deer should appear south of the camera (smaller latitude, same longitude).